# Generation: base, SFT and DPO outputs on the fixed eval sets

**Before running**
1. Push the repo (`eval_sets.py`, `generate.py`, `pairs.py`, `eval_sets/alpaca_eval_300_seed42.json`) to GitHub.
2. Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**.
3. **Add Input** → your saved **SFT** notebook version *and* your saved **DPO** notebook version. The notebook finds `outputs/sft/best`, `outputs/dpo_hh/best` and `outputs/dpo_ultrafeedback/best` inside them automatically.
4. Private repo: Kaggle secret `GITHUB_TOKEN`, as in the training notebooks.

**What it does**
- Every model uses the same prompt template and the same greedy decoding settings (one `DECODING` dict below), so the judge compares models, not decoding policies.
- AlpacaEval jobs run on `cuda:0` and HH jobs on `cuda:1` at the same time.
- Each generation file is rewritten after every batch. If the session dies, attach this notebook's saved output as an input and rerun: finished prompts are skipped.
- At the end it builds the blinded DPO-vs-SFT pairwise files for the judge notebook.

**How to use**
- **Sanity check (manual §12):** `MODE = "sanity"`, run all cells interactively (~3 min). Read the printed responses: right checkpoint, sensible format, answers end cleanly.
- **Full:** `MODE = "full"`, then **Save Version → Save & Run All (Commit)** (~30–45 min).

In [ ]:
# ---- Configuration ----
MODE = "sanity"                   # "sanity" (3 prompts per set, printed) or "full"

REPO_URL = "https://github.com/HafsaKamran/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/small-model-alignment"

# One decoding policy for every model (report these in §14.2). Greedy, per the manual;
# a mild repetition penalty because the 0.5B models otherwise loop on long answers.
DECODING = {"max_new_tokens": 512, "repetition_penalty": 1.1, "no_repeat_ngram_size": 0}
BATCH_SIZE = 16

# Also run each DPO model on the *other* track's eval set (e.g. HH-DPO on AlpacaEval).
# Not required, but cheap, and it shows whether one kind of alignment costs the other (alignment tax).
CROSS_EVAL = True

LIMIT = 3 if MODE == "sanity" else None
GEN_DIR = f"{REPO_DIR}/outputs/" + ("generations_sanity" if MODE == "sanity" else "generations")
EVAL_DIR = f"{REPO_DIR}/outputs/eval"
assert MODE in ("sanity", "full"), MODE
print(MODE, GEN_DIR, DECODING)

## 1. GPUs, repo, dependencies

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "needs GPU T4 x2"

In [ ]:
import os, subprocess

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
    print("Using GITHUB_TOKEN secret")
except Exception:
    print("No GITHUB_TOKEN secret; cloning as a public repo")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], check=True)

%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps || echo "editable install failed; fine, we run with python -m from the repo root"
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
!PYTHONPATH=. python -m pytest -q tests/

## 2. Find the checkpoints

In [ ]:
import glob

def find_checkpoint(run):
    found = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
                   if p.replace("\\", "/").endswith(f"/outputs/{run}/best/config.json"))
    assert found, f"No .../outputs/{run}/best found. Add the notebook that trained it via Add Input."
    if len(found) > 1:
        print(f"  several {run} checkpoints, using the first:", found)
    ckpt = found[0]
    assert any(os.path.exists(f"{ckpt}/{f}") for f in ("model.safetensors", "model.safetensors.index.json")), ckpt
    return ckpt

# key -> (generator name stored in every row, checkpoint)
MODELS = {
    "base": ("qwen2.5-0.5b-base", "Qwen/Qwen2.5-0.5B"),
    "sft": ("qwen2.5-0.5b-sft", find_checkpoint("sft")),
    "dpo_hh": ("qwen2.5-0.5b-dpo-hh", find_checkpoint("dpo_hh")),
    "dpo_ultrafeedback": ("qwen2.5-0.5b-dpo-ultrafeedback", find_checkpoint("dpo_ultrafeedback")),
}
for key, (name, path) in MODELS.items():
    print(f"{key:18} {name:32} {path}")

## 3. Resume from a previous run (if attached)

If a saved version of *this* notebook is attached as an input, its finished generations are copied in, and `generate.py` skips every prompt that is already done.

In [ ]:
import shutil

previous = [d for d in glob.glob("/kaggle/input/**/outputs/" + os.path.basename(GEN_DIR), recursive=True) if os.path.isdir(d)]
for d in previous:
    print("resuming from", d)
    shutil.copytree(d, GEN_DIR, dirs_exist_ok=True)
if not previous:
    print("no previous generations attached; starting fresh")

## 4. Generate

AlpacaEval jobs on `cuda:0`, HH jobs on `cuda:1`, in parallel. Each GPU works through its list one model at a time. Logs stream below; if any job fails, the cell raises.

In [ ]:
import sys, threading, time

JOBS = {
    "alpaca": ["base", "sft", "dpo_ultrafeedback"] + (["dpo_hh"] if CROSS_EVAL else []),
    "hh":     ["base", "sft", "dpo_hh"] + (["dpo_ultrafeedback"] if CROSS_EVAL else []),
}
DEVICES = {"alpaca": "cuda:0", "hh": "cuda:1"}

def gen_path(eval_set, key):
    return f"{GEN_DIR}/{eval_set}/{key}.json"

def command(eval_set, key):
    name, path = MODELS[key]
    cmd = [sys.executable, "-m", "src.alignment.generate", "--model_path", path, "--generator", name,
           "--eval_set", eval_set, "--output_path", gen_path(eval_set, key), "--device", DEVICES[eval_set],
           "--batch_size", str(BATCH_SIZE)]
    for k, v in DECODING.items():
        cmd += [f"--{k}", str(v)]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    return cmd

env = dict(os.environ, TOKENIZERS_PARALLELISM="false")
failures, logs = [], {}

def worker(eval_set):
    os.makedirs(f"{GEN_DIR}/{eval_set}", exist_ok=True)
    with open(logs[eval_set], "w") as log:
        for key in JOBS[eval_set]:
            log.write(" ".join(command(eval_set, key)) + "\n"); log.flush()
            if subprocess.run(command(eval_set, key), cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT).returncode:
                failures.append((eval_set, key))

threads = []
for eval_set in JOBS:
    logs[eval_set] = f"{GEN_DIR}/{eval_set}.log"
    os.makedirs(GEN_DIR, exist_ok=True)
    open(logs[eval_set], "w").close()
    threads.append(threading.Thread(target=worker, args=(eval_set,)))
    threads[-1].start()

printed = {eval_set: 0 for eval_set in JOBS}
def show_new_lines():
    for eval_set, path in logs.items():
        lines = [l for l in open(path).read().splitlines() if "Warning" not in l and "warn(" not in l]
        for line in lines[printed[eval_set]:]:
            print(f"[{DEVICES[eval_set]}] {line}", flush=True)
        printed[eval_set] = len(lines)

while any(t.is_alive() for t in threads):
    show_new_lines()
    time.sleep(10)
show_new_lines()
if failures:
    raise RuntimeError(f"failed jobs: {failures}; see the log lines above")
print("all generation jobs finished")

## 5. Check the outputs

In [ ]:
import json, textwrap

def load(eval_set, key):
    return json.load(open(gen_path(eval_set, key), encoding="utf-8"))

print(f"{'eval set':8} {'model':18} {'n':>4} {'EOS':>5} {'cut':>4} {'empty':>5} {'max-len':>7} {'mean tok':>8}")
for eval_set, keys in JOBS.items():
    for key in keys:
        rows = load(eval_set, key)
        n = len(rows)
        print(f"{eval_set:8} {key:18} {n:>4} {sum(r['stopped_at_eos'] for r in rows):>5} "
              f"{sum(r['cut_at_stop_marker'] for r in rows):>4} {sum(not r['output'] for r in rows):>5} "
              f"{sum(r['n_new_tokens'] >= DECODING['max_new_tokens'] for r in rows):>7} "
              f"{sum(r['n_new_tokens'] for r in rows) / n:>8.0f}")
print("\nEOS = stopped by itself; cut = ran into a new '### Instruction:' turn; max-len = hit max_new_tokens.")
print("Trained models should stop at EOS almost always. Many max-len rows suggest repetition loops.")

# Side-by-side responses for a few prompts (all prompts in sanity mode)
for eval_set, keys in JOBS.items():
    outputs = {key: load(eval_set, key) for key in keys}
    for i in range(min(3, len(outputs[keys[0]]))):
        print("\n" + "=" * 100 + f"\n[{eval_set}] {outputs[keys[0]][i]['instruction'][:300]}")
        for key in keys:
            text = outputs[key][i]["output"]
            print(f"\n--- {key} ({outputs[key][i]['n_new_tokens']} tokens)\n" +
                  textwrap.shorten(text, 700, placeholder=" [...]") if text else f"\n--- {key}: <empty>")

## 6. Build the blinded pairwise files

The required comparisons are **UltraFeedback-DPO vs SFT on AlpacaEval** and **HH-DPO vs SFT on HH**. The others are for your own analysis (cross-track alignment tax, and how much SFT improved on the base model). Response order is a seeded per-prompt coin flip; model identities sit under `hidden` and never reach the judge.

In [ ]:
PAIRS = [  # (eval set, candidate, baseline)
    ("alpaca", "dpo_ultrafeedback", "sft"),   # required
    ("hh", "dpo_hh", "sft"),                  # required
    ("alpaca", "sft", "base"),                # optional: what SFT bought over the base model
]
if CROSS_EVAL:
    PAIRS += [("alpaca", "dpo_hh", "sft"), ("hh", "dpo_ultrafeedback", "sft")]

if MODE == "full":
    os.makedirs(EVAL_DIR, exist_ok=True)
    for eval_set, cand, base in PAIRS:
        out = f"{EVAL_DIR}/{eval_set}_{cand}_vs_{base}_pairs.jsonl"
        subprocess.run([sys.executable, "-m", "src.alignment.pairs", "--model_path", gen_path(eval_set, cand),
                        "--baseline_path", gen_path(eval_set, base), "--output_path", out], cwd=REPO_DIR, check=True)
else:
    print("Sanity mode: skipping pair files. Read the responses above, then switch MODE to 'full'.")

## 7. Package outputs (full run only)

In [ ]:
import zipfile

if MODE == "full":
    zip_path = "/kaggle/working/generation_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in (GEN_DIR, EVAL_DIR):
            for path in glob.glob(f"{root}/**/*", recursive=True):
                if os.path.isfile(path):
                    zf.write(path, os.path.relpath(path, f"{REPO_DIR}/outputs"))
    print("wrote", zip_path, f"{os.path.getsize(zip_path)/1e6:.1f} MB")
    !ls -la {EVAL_DIR}
else:
    print("Sanity mode: skipping zip")